# 02 Company profiles (facets)

**What this does.** Describes each company's business model in the same fields, so companies can be compared.
- **General facets (current):** 12 facets (F1 offering … F12 regulatory exposure) filled by Claude Sonnet 5.5 through
  the Claude Code CLI (Matthew and Richard, 2026-11-04), prompt `2026-10-25k`. First pass: company name, website and
  model knowledge. A second pass reads the saved website pages only for facets still unknown; a third checks
  multi-business splits. The model never sees SIC codes, tickers or financial fields. Company size (F9) is filled but
  left out of B3.
- **Sector facets (Fiona):** sector-specific fields; her sub-sector feeds the current B3 variant.
- **Industry facets (Barbara):** SIC2 modules from her dictionary; not in B3.
- **Divestiture corrections (Yiming):** a copy of the general facets with activities the companies have exited
  removed (42 companies, flagged `divestiture_corrected`; built by v1's `integration_sources.py`). Kept, not the
  default: Barbara's 293 pairs and the current B3 list use the uncorrected fills.

**Saved outputs:** `outputs-local/v1a/02_profiles/` (tables). The raw model answers stay in `outputs-local/v1/`
(linked in `seed_manifest.json`).

**Changelog**
- 2026-11-18: created from `v1/notebooks/comp_universe_and_facets.ipynb`, `facet_fill_runs.ipynb`,
  `business_structure_eval.ipynb` and `business_split_refinement_eval.ipynb`. The three tables rebuild exactly
  from the saved raw answers.
- 2026-11-19: v1 `comp_universe_and_facets.ipynb` tables ported on the current Sonnet fills: the facet definitions
  (3.1) and section 3.6 (coverage, flags, categories, review list, example profiles). `facet_fill_runs.ipynb`
  (Haiku/Codex fill runs, API checks, costs) stays in v1 as the record.


In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

## 1. Current setting (`v1a/current.toml`)

In [2]:
pd.Series(cur["profiles"])

general                                          outputs-local/v1a/02_profiles/facets_sonnet/tables/facets.csv
general_prompt_version                                                                             2026-10-25k
sector                             outputs-local/v1a/02_profiles/sector_facets_sonnet/tables/sector_facets.csv
industry                  outputs-local/v1a/02_profiles/industry_facets_sonnet/tables/industry_facets_long.csv
corrected_general                                    outputs-local/v1a/02_profiles/facets_sonnet_corrected.csv
website_pages                                                           outputs-local/website_scrape/page_text
dtype: object

## 2. Inputs

In [3]:
nb.inputs([cur["profiles"][k] for k in ("general", "sector", "industry", "corrected_general")])

,input,rows,sha256,status
0,outputs-local/v1a/02_profiles/facets_sonnet/tables/facets.csv,533,8619ecf21c32,matches seed
1,outputs-local/v1a/02_profiles/sector_facets_sonnet/tables/sector_facets.csv,533,a7139d71efd6,matches seed
2,outputs-local/v1a/02_profiles/industry_facets_sonnet/tables/industry_facets_long.csv,5331,8b7b986d8697,matches seed
3,outputs-local/v1a/02_profiles/facets_sonnet_corrected.csv,533,ba9a51e7b323,matches seed


## 3. Results

### 3.1 General facets: what was filled

In [4]:
# the twelve facets (v1 comp_universe_and_facets.ipynb 3.1)
import facets as fc
pd.DataFrame([{"ID": fc.FACET_ID[f.name], "Facet": f.label, "What it captures": f.captures, "Example values": f.examples,
               "Categories": f"{f.kind}: {', '.join(f.categories)}" if f.categories else ""}
              for f in fc.FACETS]).set_index("ID")

,Facet,What it captures,Example values,Categories
ID,,,,
F1,Core offering,"What the company sells, and to whom",Designs and manufactures rooftop HVAC units for commercial buildings,
F2,Primary products,The main products it sells,HR software; medical devices; hot-mix asphalt,
F3,Primary services,The main services it provides,consulting; freight logistics; payroll processing,
F4,Customer segment,Who buys its products or services,"B2B: mechanical contractors, commercial building owners; B2G: school districts","any of: B2B, B2C, B2G"
F5,End markets,The industries it serves,healthcare; retail; manufacturing,
F6,Revenue model,How it earns money,product sales: equipment sold through dealers; service fees: installation and repair,"any of: product sales, subscription, transaction or usage fees, licensing or royalties..."
F7,Value chain position,Its role in the industry,"OEM / manufacturer: builds HVAC units from purchased components, sold to contractors","any of: raw materials, component supplier, OEM / manufacturer, brand owner, distributo..."
F8,Geography served,Where it sells or operates,"US regional: Alabama, Florida, Georgia and the Carolinas","one of: US regional, US national, North America, global"
F9,Company size,How large it is within its own industry,mid-sized,"one of: small, mid-sized, large, among the largest"


In [5]:
import facets as fc, facet_weighting as fw
general = pd.read_csv(paths.path(cur["profiles"]["general"]), dtype=str).fillna("")
names = list(fc.FACET_ID)
used = set(fw.similarity_facets())
status = pd.DataFrame({n: general[f"{n}_status"].value_counts() for n in names}).T.fillna(0).astype(int)
status.insert(0, "id", [fc.FACET_ID[n] for n in names])
status["in B3"] = ["yes" if n in used else "no" for n in names]
print(f"{general['IQ_COMPANY_ID'].nunique()} companies, {len(general)} business rows; prompt {general['prompt_version'].unique().tolist()}; "
      f"status {general['llm_status'].value_counts().to_dict()}")
status

533 companies, 533 business rows; prompt ['2026-10-25k']; status {'ok': 533}


,id,filled,not_applicable,in B3
offering_description,F1,533,0,yes
primary_products,F2,419,114,yes
primary_services,F3,414,119,yes
customer_segment,F4,533,0,yes
end_markets,F5,533,0,yes
revenue_model,F6,533,0,yes
value_chain_position,F7,533,0,yes
geography_served,F8,533,0,yes
company_size,F9,533,0,no
specialization,F10,533,0,yes


### 3.2 Where the values come from

Each facet value records its source: model knowledge, or the website lines it cites. The website pass only runs
for facets the first pass left unknown.

In [6]:
src = pd.DataFrame({n: general[f"{n}_source"] for n in names})
kind = src.apply(lambda col: col.map(lambda s: "website" if "http" in s else ("model knowledge" if "model knowledge" in s else "none")))
by_source = kind.apply(lambda c: c.value_counts()).T.fillna(0).astype(int)
display(by_source)
print("share of filled values from model knowledge only:",
      f"{(kind == 'model knowledge').sum().sum() / (kind != 'none').sum().sum():.1%}")
print("companies with at least one website-sourced value:", int((kind == "website").any(axis=1).sum()))

,model knowledge,none,website
offering_description,523,0,10
primary_products,526,3,4
primary_services,523,0,10
customer_segment,523,0,10
end_markets,525,0,8
revenue_model,524,1,8
value_chain_position,525,0,8
geography_served,525,0,8
company_size,527,2,4
specialization,524,0,9


share of filled values from model knowledge only: 98.5%
companies with at least one website-sourced value: 10


### 3.3 Sector facets (Fiona)

In [7]:
sector = pd.read_csv(paths.path(cur["profiles"]["sector"]), dtype=str).fillna("")
print(f"{len(sector)} companies; status {sector['llm_status'].value_counts().to_dict()}; prompt {sector['prompt_version'].unique().tolist()}")
print("sub-sector (used by B3):", sector["sub_sector_status"].value_counts().to_dict())
sector[["IQ_COMPANY_ID", "sector", "sub_sector"]].head(8)

533 companies; status {'ok': 533}; prompt ['2026-10-27b']
sub-sector (used by B3): {'stated': 533}


,IQ_COMPANY_ID,sector,sub_sector
0,IQ100918,energy_resources,"integrated electric and gas utility, Pacific Northwest"
1,IQ100935,construction_real_estate,"commercial mechanical, electrical and plumbing contracting; building service and maint..."
2,IQ1032802,software,investor communications and capital markets post-trade fintech; wealth management tech...
3,IQ105613224,manufacturing,Industrial electric heat tracing and process heating; industrial boilers
4,IQ1064486,manufacturing,payment card manufacturing; instant issuance and card fulfillment services; prepaid ca...
5,IQ1067110,business_services,digital learning courseware and education publishing; medical education
6,IQ107405,construction_real_estate,single-family homebuilding; mortgage and title services; build-to-rent homes
7,IQ108424,distribution_transport,industrial MRO supplies distribution; metalworking tools and supplies


### 3.4 Industry facets (Barbara's SIC2 modules)

In [8]:
industry = pd.read_csv(paths.path(cur["profiles"]["industry"]), dtype=str)
share = industry["status"].value_counts(normalize=True).mul(100).round(1)
print(f"{len(industry)} facet rows for {industry['IQ_COMPANY_ID'].nunique()} companies; status %:", share.to_dict())
industry.groupby("industry")["status"].value_counts(normalize=True).unstack().fillna(0).mul(100).round(0).head(12)

5331 facet rows for 533 companies; status %: {'stated': 48.2, 'not_applicable': 28.6, 'unknown': 23.2}


status,not_applicable,stated,unknown
industry,,,
Agricultural Services,0.0,80.0,20.0
Amusement & Recreation,27.0,45.0,28.0
Apparel & Finished Textile Products,3.0,87.0,10.0
Building Construction Contractors and Operative Builders,3.0,68.0,29.0
Business Services,45.0,34.0,20.0
Chemicals & Allied Products,19.0,62.0,19.0
Communications,20.0,54.0,27.0
Depository Institutions,100.0,0.0,0.0
Eating & Drinking Places,0.0,42.0,58.0


### 3.5 Divestiture corrections (Yiming; kept, not the default)

In [9]:
corrected = pd.read_csv(paths.path(cur["profiles"]["corrected_general"]), dtype=str).fillna("")
key = ["IQ_COMPANY_ID", "business_id"]
a, b = general.set_index(key).sort_index(), corrected.set_index(key).sort_index()[general.columns.drop(key)]
diff = (a != b)
changes = [(i[0], c, a.at[i, c][:70], b.at[i, c][:70]) for i, row in diff.iterrows() for c in row.index[row]]
changes = pd.DataFrame(changes, columns=["IQ_COMPANY_ID", "column", "current (Sonnet)", "corrected"])
universe = pd.read_csv(paths.path(cur["universe"]["universe"]), dtype=str).set_index("IQ_COMPANY_ID")
changes.insert(1, "ticker", changes["IQ_COMPANY_ID"].map(universe["target_ciq_ticker"]))
value_changes = changes[~changes["column"].str.endswith(("_source", "_flags", "_status", "_category"))]
print(f"{value_changes['IQ_COMPANY_ID'].nunique()} companies, {len(value_changes)} value fields changed; by facet:",
      value_changes["column"].value_counts().to_dict())
value_changes.head(12)

42 companies, 116 value fields changed; by facet: {'offering_description': 28, 'specialization': 27, 'primary_products': 24, 'end_markets': 22, 'primary_services': 15}


,IQ_COMPANY_ID,ticker,column,current (Sonnet),corrected
0,IQ121509,ZD,offering_description,Operates a portfolio of digital media brands and online properties cov,Operates a portfolio of digital media brands and online properties cov
1,IQ121509,ZD,primary_products,digital media websites; VPN and cybersecurity software; email marketin,digital media websites; VPN and cybersecurity software; email marketin
2,IQ121509,ZD,primary_services,digital advertising; lead generation; performance marketing; email and,digital advertising; lead generation; performance marketing; email and
3,IQ121509,ZD,specialization,"related lines: digital media, performance marketing, cybersecurity and","related lines: digital media, performance marketing, cybersecurity and"
4,IQ130863,FSTR,offering_description,Manufactures and supplies engineered products for rail and transit inf,Manufactures and supplies engineered products for rail and transit inf
5,IQ130863,FSTR,primary_products,rail track components and fasteners; friction management systems; prec,friction management systems; precast concrete products; protective coa
6,IQ168154,AIR,primary_products,aircraft parts and components; engineered pallets and shelters for mil,aircraft parts and components; engineered pallets and shelters for mil
7,IQ170928919,TRU,end_markets,financial services; insurance; healthcare; telecommunications; retail;,financial services; insurance; telecommunications; retail; government
8,IQ18671,ALB,offering_description,"Produces lithium compounds for batteries, bromine-based flame retardan","Produces lithium compounds for batteries, bromine-based flame retardan"
9,IQ18671,ALB,primary_products,lithium carbonate and hydroxide; bromine and brominated flame retardan,lithium carbonate and hydroxide; bromine and brominated flame retardan


### 3.6 The filled facets, in full (from v1 `comp_universe_and_facets.ipynb` 3.3)

Share filled / not applicable / unknown per facet, the flags the fill raised, the categories chosen, the companies
flagged for manual review and why, and example profiles to read in full (the first company for each review reason,
then the first with nothing to review).

In [10]:
# v1 comp_universe_and_facets.ipynb 3.3 on the current fills (Sonnet, prompt in current.toml); v1 read the Haiku fills
universe = pd.read_csv(paths.path(cur["universe"]["universe"]), dtype=str)
ran = general[(general["prompt_version"] == cur["profiles"]["general_prompt_version"]) & (general["llm_status"] == "ok")]
print(f"facets version {cur['profiles']['general_prompt_version']}: {ran['IQ_COMPANY_ID'].nunique()} of "
      f"{general['IQ_COMPANY_ID'].nunique()} universe companies filled, {len(ran)} business profiles")


facets version 2026-10-25k: 533 of 533 universe companies filled, 533 business profiles


In [11]:
if ran.empty:
    print("no companies filled with the current facets yet")
else:
    coverage = pd.DataFrame({f.name: ran[f"{f.name}_status"].value_counts(normalize=True)
                             for f in fc.FACETS}).T.rename(index=lambda n: f"{fc.FACET_ID[n]} {n}").fillna(0)
    coverage = coverage.reindex(columns=["filled", "not_applicable", "unknown"], fill_value=0)
    display((coverage * 100).round(1))

,filled,not_applicable,unknown
F1 offering_description,100.0,0.0,0
F2 primary_products,78.6,21.4,0
F3 primary_services,77.7,22.3,0
F4 customer_segment,100.0,0.0,0
F5 end_markets,100.0,0.0,0
F6 revenue_model,100.0,0.0,0
F7 value_chain_position,100.0,0.0,0
F8 geography_served,100.0,0.0,0
F9 company_size,100.0,0.0,0
F10 specialization,100.0,0.0,0


In [12]:
if not ran.empty:
    flags = pd.DataFrame({f.name: ran[f"{f.name}_flags"].str.split("; ").explode().replace("", pd.NA).value_counts()
                          for f in fc.FACETS}).T.rename(index=lambda n: f"{fc.FACET_ID[n]} {n}").fillna(0).astype(int)
    display(flags) if not flags.empty else print("no flags")

,website_without_quote
F1 offering_description,0
F2 primary_products,0
F3 primary_services,0
F4 customer_segment,0
F5 end_markets,0
F6 revenue_model,1
F7 value_chain_position,0
F8 geography_served,0
F9 company_size,2
F10 specialization,0


In [13]:
for f in fc.FACETS if not ran.empty else []:
    if f.kind != "text":
        counts = ran[f"{f.name}_category"].str.split("; ").explode().replace("", pd.NA).value_counts()
        print(f"{f.name}: " + ", ".join(f"{k} {v}" for k, v in counts.items()))

customer_segment: B2B 504, B2C 244, B2G 172
revenue_model: service fees 328, product sales 303, subscription 127, project or contract 86, transaction or usage fees 73, commissions or management fees 62, licensing or royalties 59, rental or leasing 58, interest or premiums 32, advertising 31, other 28, healthcare reimbursement 21
value_chain_position: service provider 217, OEM / manufacturer 187, software / platform 131, asset operator 125, brand owner 107, financial intermediary 72, distributor 35, retailer 28, component supplier 20, raw materials 11, other 6
geography_served: global 318, US national 105, US regional 57, North America 53
company_size: large 240, mid-sized 152, among the largest 93, small 48
specialization: related lines 482, niche specialist 40, diversified 11
distribution_channels: direct sales 442, online 213, distributors / dealers 143, own locations 133, partners / resellers 128, agents / brokers 59, third-party retail 52, other 23
regulatory_exposure: medium 308, 

In [14]:
review = ran[ran["needs_review"] == "True"].merge(universe[["IQ_COMPANY_ID", "target_ciq_ticker"]], on="IQ_COMPANY_ID")
print(f"{review['IQ_COMPANY_ID'].nunique()} of {ran['IQ_COMPANY_ID'].nunique()} companies need the manual review")
display(review["review_reasons"].str.split("; ").explode().value_counts().to_frame("profiles"))
review.set_index("target_company_name")[["target_ciq_ticker", "business_id", "review_reasons", "notes"]]

23 of 533 companies need the manual review


,profiles
review_reasons,
pages_mention_divestiture,13
described_from_pages_only,10
facet_flagged,2


,target_ciq_ticker,business_id,review_reasons,notes
target_company_name,,,,
AdaptHealth Corp.,AHCO,B1,pages_mention_divestiture,
"American Financial Group, Inc.",AFG,B1,pages_mention_divestiture,
Andersen Group Inc.,ANDG,B1,described_from_pages_only,"The name ""Andersen Group Inc."" does not clearly match the website andersen.com, which ..."
Bioventus Inc.,BVS,B1,pages_mention_divestiture,
Cardinal Infrastructure Group Inc.,CDNL,B1,described_from_pages_only,Company not recognized from name and website alone.
Chemed Corporation,CHE,B1,pages_mention_divestiture,
Covista Inc.,CVSA,B1,described_from_pages_only,I can't reliably identify what the company does today.
"CSW Industrials, Inc.",CSW,B1,pages_mention_divestiture,
"Englewood Lab, Inc.",A950140,B1,described_from_pages_only,Company not recognized with confidence from name and website alone.


In [15]:
# Examples to read in full, by a fixed rule (not a sample): the first company
# for each review reason, then the first with nothing to review. Any universe
# tickers can be put in PROFILES instead.
ticker = universe.set_index("IQ_COMPANY_ID")["target_ciq_ticker"]
by_reason = (ran[ran["needs_review"] == "True"].assign(reason=lambda d: d["review_reasons"].str.split("; "))
             .explode("reason").groupby("reason")["IQ_COMPANY_ID"].first())
clean = ran.loc[ran["needs_review"] == "False", "IQ_COMPANY_ID"]
PROFILES = list(dict.fromkeys([*ticker.loc[by_reason].tolist(), *ticker.loc[clean.iloc[:1]].tolist()]))

def fill_rate(rows: pd.DataFrame) -> pd.Series:
    """Share of the twelve facets filled or not applicable, per business profile."""
    return pd.concat([rows[f"{f.name}_status"].ne("unknown") for f in fc.FACETS], axis=1).mean(axis=1)

def profile(row: pd.Series) -> pd.DataFrame:
    return pd.DataFrame([{"ID": fc.FACET_ID[f.name], "facet": f.label, "status": row[f"{f.name}_status"],
                          "value": row[f.name], "source": row[f"{f.name}_source"]}
                         for f in fc.FACETS]).set_index("ID")

# every business profile, numbered: each universe company is a target in turn and a candidate comp for the others
filled = ran.merge(universe[["IQ_COMPANY_ID", "target_ciq_ticker"]], on="IQ_COMPANY_ID").reset_index(drop=True)
profiles = pd.DataFrame({"target": filled["target_company_name"] + " (" + filled["target_ciq_ticker"] + ")",
                         "business": filled["business_id"] + ": " + filled["business"],
                         "fill %": (fill_rate(filled) * 100).round(1)})
profiles.index = pd.RangeIndex(1, len(profiles) + 1, name="#")
if not profiles.empty:
    print(f"fill rate: {fill_rate(filled).mean():.1%} of facets over {len(profiles)} business profiles; "
          f"{(profiles['fill %'] == 100).sum()} profiles complete. Profiles with an unknown facet:")
    display(profiles[profiles["fill %"] < 100])

for t in PROFILES:
    name = universe.loc[universe["target_ciq_ticker"] == t, "target_company_name"].iloc[0]
    rows = profiles[filled["target_ciq_ticker"].eq(t).to_numpy()]
    if rows.empty:
        print(f"{name} ({t}): not filled with the current facets yet")
    for i in rows.index:
        reasons = filled.loc[i - 1, "review_reasons"]
        print(f"{name} ({t}): {'review: ' + reasons if reasons else 'nothing to review'}")
        display(rows.loc[[i]])
        display(profile(filled.loc[i - 1]))

fill rate: 100.0% of facets over 533 business profiles; 533 profiles complete. Profiles with an unknown facet:


,target,business,fill %
#,,,


Andersen Group Inc. (ANDG): review: described_from_pages_only


,target,business,fill %
#,,,
32,Andersen Group Inc. (ANDG),B1: tax advisory and consulting firm,100.0


,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Provides tax compliance and planning, valuation, and financial and business consulting...",https://andersen.com/; https://andersen.com/who-we-are
F2,Primary products,not_applicable,,model knowledge
F3,Primary services,filled,tax compliance and consulting; valuation services; financial advisory; private wealth ...,https://andersen.com/who-we-are
F4,Customer segment,filled,"B2B: businesses, alternative investment funds and fund managers, family offices; B2C: ...",https://andersen.com/who-we-serve; https://andersen.com/
F5,End markets,filled,alternative investment funds; private wealth; corporate businesses across industries; ...,model knowledge
F6,Revenue model,filled,"service fees: fees for tax, valuation, financial advisory and consulting engagements",https://andersen.com/who-we-are
F7,Value chain position,filled,"service provider: professional advisors delivering tax, valuation and consulting servi...",model knowledge
F8,Geography served,filled,global: US offices plus Andersen Global association of independent member firms in man...,https://andersen.com/who-we-are; https://andersen.com/
F9,Company size,filled,large,model knowledge


Legacy Education Inc. (LGCY): review: described_from_pages_only; facet_flagged


... (4 outputs left out)

,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Provides home medical equipment, medical supplies and related services to patients in ...",model knowledge
F2,Primary products,filled,CPAP and sleep therapy devices and supplies; continuous glucose monitors and insulin p...,model knowledge
F3,Primary services,filled,home delivery of medical equipment; patient setup and training; resupply programs; res...,model knowledge
F4,Customer segment,filled,"B2C: patients using home medical equipment, billed mostly through insurance; B2B: refe...",model knowledge
F5,End markets,filled,home healthcare; sleep apnea; diabetes care; respiratory care; mobility and chronic di...,model knowledge
F6,Revenue model,filled,"healthcare reimbursement: payments from Medicare, Medicaid and commercial insurers for...",model knowledge
F7,Value chain position,filled,"service provider: buys equipment and supplies from manufacturers, delivers to patients...",model knowledge
F8,Geography served,filled,US national: Operates branches across most US states,model knowledge
F9,Company size,filled,among the largest,model knowledge


AAR Corp. (AIR): nothing to review


,target,business,fill %
#,,,
1,AAR Corp. (AIR),B1: aviation services and aftermarket support,100.0


,facet,status,value,source
ID,,,,
F1,Core offering,filled,"Provides aviation aftermarket services to commercial airlines, governments and defense...",model knowledge
F2,Primary products,filled,aircraft parts and components; engineered pallets and shelters for military mobility; ...,model knowledge
F3,Primary services,filled,"aircraft maintenance, repair and overhaul; parts supply chain and distribution; compon...",model knowledge
F4,Customer segment,filled,"B2B: commercial airlines, cargo carriers, aircraft lessors and original equipment manu...",model knowledge
F5,End markets,filled,commercial aviation; defense and military aviation; government aviation,model knowledge
F6,Revenue model,filled,"service fees: aircraft maintenance, repair, overhaul and component repair; product sal...",model knowledge
F7,Value chain position,filled,"service provider: maintenance, repair and overhaul using purchased parts, serving airl...",model knowledge
F8,Geography served,filled,"global: United States, Europe, Asia, Middle East and other international markets",model knowledge
F9,Company size,filled,large,model knowledge


## 4. Checks against the saved v1 numbers

In [16]:
import contextlib, io, shutil, tempfile
import fill_facets as ff, fill_sector_facets as fs, fill_industry_facets as fi
rebuild = [("general", "facets_sonnet", lambda d: ff.write_tables(d), "tables/facets.csv", cur["profiles"]["general"]),
           ("sector", "sector_facets_sonnet", lambda d: fs.write_tables(fs.output_paths(d)), "tables/sector_facets.csv", cur["profiles"]["sector"]),
           ("industry", "industry_facets_sonnet", lambda d: fi.write_tables(fi.output_paths(d)), "tables/industry_facets_long.csv", cur["profiles"]["industry"])]
with tempfile.TemporaryDirectory() as tmp:               # rebuild from the raw answers in the v1 record (read only)
    for name, folder, write, table, seeded in rebuild:
        d = Path(tmp) / folder
        shutil.copytree(paths.V1_OUTPUTS / folder / "profiles", d / "profiles")
        with contextlib.redirect_stdout(io.StringIO()):    # the writers print temp paths
            write(d)
        x, y = pd.read_csv(d / table, dtype=str).fillna(""), pd.read_csv(paths.path(seeded), dtype=str).fillna("")
        checks.add(f"{name} table rebuilds exactly from the raw answers", list(x.columns) == list(y.columns) and x.equals(y), f"{x.shape}")
checks.add("general: 533 companies, all ok, prompt 2026-10-25k", general["IQ_COMPANY_ID"].nunique() == 533
           and general["llm_status"].eq("ok").all() and general["prompt_version"].eq(cur["profiles"]["general_prompt_version"]).all())
checks.add("sector: 533 companies, sub-sector stated for all", len(sector) == 533 and sector["sub_sector_status"].eq("stated").all())
checks.add("industry: 5,331 rows (v1 handoff 2026-11-04)", len(industry) == 5331, str(share.to_dict()))
flagged = set(corrected.loc[corrected["divestiture_corrected"] == "True", "IQ_COMPANY_ID"])
checks.add("corrections are exactly the 42 flagged companies", set(value_changes["IQ_COMPANY_ID"]) == flagged and len(flagged) == 42,
           f"{len(value_changes)} value fields")
checks.show()

7/7 checks pass


,check,result,detail
0,general table rebuilds exactly from the raw answers,pass,"(533, 74)"
1,sector table rebuilds exactly from the raw answers,pass,"(533, 199)"
2,industry table rebuilds exactly from the raw answers,pass,"(5331, 12)"
3,"general: 533 companies, all ok, prompt 2026-10-25k",pass,
4,"sector: 533 companies, sub-sector stated for all",pass,
5,"industry: 5,331 rows (v1 handoff 2026-11-04)",pass,"{'stated': 48.2, 'not_applicable': 28.6, 'unknown': 23.2}"
6,corrections are exactly the 42 flagged companies,pass,116 value fields


## 5. History (v1)

- **Haiku fills** (`outputs-local/v1/facets/`, 2026-10-25) were the v0/v1 profiles. Sonnet replaced them on
  2026-11-04; Yifan's check found Haiku still described businesses CSL, AFG and AZZ had sold.
- **Business split pilot** (`business_split.py`, `business_split_refinement.py`): source-checked business groups
  for 40 companies; kept as metadata, not adopted as the peer rule (v1 handoff 2026-10-28).
- **Facet verification** (`scripts/verification/`): evidence retrieval and checks of facet claims (Yiming).
- **Alternatives tried in the 11-10 integration study** (stay in v1): GPT-6.1 Sol fills, cited-only fills,
  Yifan's Orbis-aligned fields. No measured gain over these profiles; see `09_analysis`.